# Modèle de gravité — coût conditionnel

Le modèle est entraîné uniquement sur les sinistres positifs. Nous cherchons une **espérance de coût**, car la prime finale est évaluée par une erreur quadratique. Une médiane ou une exponentielle naïve de log-coût ne remplit pas automatiquement cet objectif.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'run.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pricing.experiment import load_data
train, test, development, historique = load_data()
R = ROOT / 'reports'
selection = json.loads((R / 'selection.json').read_text(encoding='utf-8'))
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})


In [2]:
claims=train.loc[train.nombre_sinistres.eq(1)]
claims.montant_sinistre.describe(percentiles=[.5,.9,.95,.99])

count     2917.000000
mean      1771.843401
std       1463.954673
min        600.600000
50%       1236.000000
90%       3336.740000
95%       4355.258000
99%       6963.118800
max      21826.960000
Name: montant_sinistre, dtype: float64

In [3]:
fig,ax=plt.subplots();ax.hist(claims.montant_sinistre,bins=60,color='#157a6e');ax.set(xlabel='Coût du sinistre',ylabel='Effectif',yscale='log',title='Asymétrie et sinistres rares de coût élevé');plt.show()

C:\Users\fourr\AppData\Local\Temp\ipykernel_24488\1682609228.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.hist(claims.montant_sinistre,bins=60,color='#157a6e');ax.set(xlabel='Coût du sinistre',ylabel='Effectif',yscale='log',title='Asymétrie et sinistres rares de coût élevé');plt.show()


## Modèles et régularisation

Référence Gamma à lien logarithmique avec ancienneté du véhicule, contrat et usage. Alternatives : Gamma enrichi avec splines ; régression de Poisson à lien log pour une moyenne positive ; Ridge sur le coût ; CatBoost et LightGBM. Les modèles flexibles sont régularisés car environ 2 900 observations positives sont disponibles.

Les sinistres extrêmes sont conservés dans l’apprentissage et l’évaluation : leur suppression réduirait artificiellement le risque estimé. Les prédictions de coût sont positives. Les plis sont construits sur tous les contrats et partagés avec la fréquence, puis seuls les sinistres du pli d’apprentissage entraînent la gravité.

In [4]:
pd.read_csv(R/'severity_cv.csv').round(4)

,model,MSE,RMSE,MAE,bias,predicted_observed_ratio
0,gamma_reference,2.249858e+06,1499.9526,932.6212,-2.2034,0.9988
1,gamma_basic_a0.01,2.253746e+06,1501.2481,932.8568,0.4295,1.0002
2,gamma_basic_a0.001,2.255160e+06,1501.7191,933.1984,0.9859,1.0006
3,cat_severity_d2,2.256194e+06,1502.0634,934.0074,-4.1947,0.9976
4,gamma_basic_a1.0,2.257800e+06,1502.5976,940.4865,-5.6557,0.9968
5,cat_severity_d4,2.258643e+06,1502.8783,932.2027,-3.9518,0.9978
6,cat_severity_d3,2.258939e+06,1502.9766,932.3477,-5.7364,0.9968
7,gamma_smooth_a1.0,2.261613e+06,1503.8660,939.8464,-6.2824,0.9964
8,gamma_smooth_a0.1,2.263917e+06,1504.6319,936.0001,-2.9875,0.9983
9,poisson_smooth_a0.1,2.267015e+06,1505.6610,938.4873,0.8823,1.0005


In [5]:
[(c,w) for c,w in zip(selection['severity_configs'],selection['choice']['severity_weights']) if w]

[({'name': 'gamma_basic_a0.01', 'kind': 'gamma', 'alpha': 0.01}, 1.0)]

## Pourquoi ne pas choisir uniquement la meilleure RMSE de gravité ?

Une amélioration sur les seuls sinistrés peut détériorer la prime sur l’ensemble des contrats. Le couple et ses mélanges sont donc sélectionnés par `mean((probabilité × coût_conditionnel − coût_réel)**2)` sur les prédictions hors pli.

In [6]:
v=pd.read_csv(R/'historical_predictions.csv');positive=v.loc[v.claim.eq(1)]
print('RMSE conditionnelle historique :',np.sqrt(np.mean((positive.montant_pred-positive.cost)**2)))
positive[['cost','montant_pred']].describe()

RMSE conditionnelle historique : 1245.7166171228562


,cost,montant_pred
count,565.000000,565.000000
mean,1799.079646,1778.179875
std,1254.261380,204.735902
min,600.670000,863.983924
25%,1010.410000,1690.994772
50%,1251.000000,1824.276119
75%,2265.020000,1916.639766
max,7068.540000,2081.087610
